# 01 — Préparation des données de consommation RTE

## Objectif

Construire une série **horaire** propre de la consommation électrique de la France
métropolitaine, à partir des données éCO2mix de RTE. C'est la **variable cible** du projet :
à 14 h le jour J, on voudra prévoir les 24 valeurs horaires du jour J+1.

## Source

- RTE, jeu de données **`eco2mix-national-cons-def`** sur la plateforme ODRÉ (opendata.reseaux-energies.fr).
- Il contient les **données définitives** jusqu'au 31/12/2024 et les **données consolidées** à partir du 01/01/2025.
- La consommation est publiée au **pas de 30 minutes**, en **MW** (c'est une puissance moyenne, pas une énergie).

## Les 11 étapes de ce notebook

1. Chargement des données
2. Vérification de la structure
3. Sélection de la consommation nationale
4. Nettoyage
5. Gestion des dates et heures
6. Agrégation à l'heure
7. Gestion des valeurs manquantes
8. Contrôles qualité
9. Exploration de la série
10. Sauvegarde des données préparées
11. Disponibilité des données et protocole à 14 h

Toutes les décisions prises ici sont reprises dans `docs/decisions.md`.

## Où est le code ?

Le code qui prépare les données est dans **`src/rte.py`** (et la règle des 14 h dans
**`src/protocole.py`**). Ce notebook **appelle** ces fonctions, une étape après l'autre, et
explique ce qu'elles font et ce qu'on observe. C'est la règle du groupe (`CONTRIBUTING.md`) :
le code réutilisable va dans `src/`, les notebooks ne font que l'appeler.

- Pour relancer toute la préparation d'un coup, sans le notebook : `python -m src.rte`.
- Pour lire le code d'une fonction ici même : `voir_code(rte.nom_de_la_fonction)`.

## 0. Préparation : bibliothèques et chemins

Je ne réécris pas les chemins à la main : je les prends dans `src/config.py`, comme ça
tout le groupe utilise exactement les mêmes dossiers et les mêmes dates.

In [ ]:
import sys
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

# Je cherche la racine du projet : c'est le premier dossier parent qui contient "src".
# Comme ça le notebook marche même s'il est lancé depuis notebooks/ ou depuis la racine.
RACINE = Path.cwd().resolve()
while not (RACINE / "src").exists() and RACINE != RACINE.parent:
    RACINE = RACINE.parent
sys.path.insert(0, str(RACINE))

import inspect

from src import config      # toutes les décisions du groupe (dates, dossiers, fuseau...)
from src import rte         # le code de préparation des données RTE
from src import protocole   # la règle des 14 h

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

FUSEAU = config.FUSEAU  # "Europe/Paris"


def voir_code(fonction):
    """Affiche le code d'une fonction de src/, pour le lire sans quitter le notebook."""
    print(inspect.getsource(fonction))


print("Racine du projet :", RACINE)
print("Période voulue   :", config.DATA_DEBUT, "->", config.DATA_FIN)
print("Fichier brut     :", rte.FICHIER_BRUT.relative_to(RACINE))
print("Fichier final    :", rte.FICHIER_PREPARE.relative_to(RACINE))

In [ ]:
# Style commun de mes graphiques : fond clair, grille discrète, traits fins.
# Une seule couleur quand il n'y a qu'une série ; trois couleurs fixes quand je compare des groupes.
BLEU, ORANGE, VERT = "#2a78d6", "#eb6834", "#1baf7a"
GRIS_TEXTE = "#52514e"

plt.rcParams.update({
    "figure.figsize": (11, 4),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": GRIS_TEXTE,
    "xtick.color": GRIS_TEXTE,
    "ytick.color": GRIS_TEXTE,
    "axes.grid": True,
    "grid.color": "#e6e5e0",
    "grid.linewidth": 0.8,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "lines.linewidth": 1.5,
})

## 1. Chargement des données

**Ce que je fais :** je télécharge **tout** le jeu éCO2mix national, tel que RTE le publie :
toutes les dates et toutes les colonnes, sans rien filtrer. Je l'enregistre tel quel dans
`data/donnees-brutes/rte/`.

**Pourquoi tout télécharger :**
- on voit la **taille réelle** du jeu de données (combien de lignes, combien de colonnes) ;
- le dossier `donnees-brutes` contient la donnée **d'origine, jamais modifiée**. Toutes les
  sélections se font ensuite dans le code (étape 3), donc on peut toujours vérifier ce qu'on a
  enlevé et pourquoi ;
- le sujet demande un code **reproductible** : n'importe qui peut relancer ce notebook et
  obtenir exactement le même fichier.

**Attention :** le fichier fait une centaine de Mo et le téléchargement prend quelques minutes.
Je ne le fais qu'une fois : si le fichier existe déjà, je le relis directement. Il n'est pas
envoyé sur GitHub (le dossier `data/donnees-brutes` est dans le `.gitignore`).

In [ ]:
# Le code du téléchargement (dans src/rte.py) : aucun filtre, ni sur les dates, ni sur les colonnes
voir_code(rte.telecharger)

In [ ]:
# Je télécharge (seulement si le fichier n'existe pas encore), puis je lis le fichier brut
chemin = rte.telecharger()
print(f"Fichier : {chemin.name} ({chemin.stat().st_size / 1e6:.0f} Mo)")

brut = rte.lire_brut(chemin)
print(f"\nLe jeu éCO2mix national contient {len(brut):,} lignes et {brut.shape[1]} colonnes.")

## 2. Vérification de la structure

**Ce que je fais :** avant de toucher aux données, je regarde à quoi elles ressemblent :
quelles colonnes existent et à quoi elles servent, quelle période est couverte, et à quelles
minutes la consommation est renseignée.

**Pourquoi :** si je me trompe sur la structure (par exemple croire que les données sont au
quart d'heure alors qu'elles sont à la demi-heure), toutes les étapes suivantes seront fausses.

In [ ]:
# Les premières lignes, toutes colonnes
brut.head(4)

In [ ]:
# Les 37 colonnes, rangées par famille.
# Pour chaque colonne : sa famille, le pourcentage de lignes remplies, et si on a le droit de
# l'utiliser pour prévoir demain à 14 h.
def famille(colonne):
    if colonne in ["perimetre", "nature", "date", "heure", "date_heure"]:
        return "Repère (où, quand, quelle version)"
    if colonne == "consommation":
        return "CIBLE : ce qu'on veut prévoir"
    if colonne.startswith("prevision"):
        return "Prévision faite par RTE"
    if colonne.startswith("ech_"):
        return "Échanges avec les pays voisins"
    if colonne == "taux_co2":
        return "Émissions de CO2"
    return "Production par filière"

utilisation = {
    "Repère (où, quand, quelle version)": "Oui : sert à situer chaque ligne",
    "CIBLE : ce qu'on veut prévoir": "Oui : c'est la cible (et son passé sert de retard)",
    "Prévision faite par RTE": "Non : comparaison possible, jamais en variable",
    "Échanges avec les pays voisins": "Non : mesurés en même temps que la consommation",
    "Émissions de CO2": "Non : calculé à partir de la production",
    "Production par filière": "Non : mesurée en même temps que la consommation",
}

colonnes = pd.DataFrame({
    "famille": [famille(c) for c in brut.columns],
    "% de lignes remplies": (brut.notna().mean() * 100).round(1).values,
}, index=brut.columns)
colonnes["utilisable à 14 h ?"] = colonnes["famille"].map(utilisation)
colonnes

**Ce que j'observe :**
- **5 colonnes repères** : le périmètre (France), la version des données (`nature`), la date
  et l'heure locales, et l'instant précis (`date_heure`).
- **1 colonne cible** : `consommation`, en MW.
- **31 autres colonnes** : prévisions de RTE, production par filière (nucléaire, gaz, éolien,
  solaire...), échanges avec les pays voisins, taux de CO2.

**Pourquoi les 31 autres colonnes sont interdites :** la production est ajustée **en temps
réel** à la consommation. Par exemple, le nucléaire de demain 18 h n'est pas connu aujourd'hui
à 14 h, au même titre que la consommation de demain 18 h. S'en servir serait une **fuite
d'information**. Les prévisions de RTE (`prevision_j1`, `prevision_j`) pourraient servir de
point de comparaison, mais jamais de variable d'entrée.

In [ ]:
# La période couverte et la version des données, année par année.
# La colonne "date" est la date en heure de Paris, écrite par RTE (format "AAAA-MM-JJ").
# Je calcule l'année à part, sans l'ajouter au tableau brut : le tableau brut reste intact.
annee = brut["date"].str[:4].rename("annee")
print("Première ligne :", brut["date"].min(), "| Dernière ligne :", brut["date"].max())
print("Périmètres présents :", list(brut["perimetre"].unique()))
pd.crosstab(annee, brut["nature"], margins=True, margins_name="Total")

**Ce que j'observe :**
- les données commencent le **1er janvier 2012** et vont jusqu'à **mi-2026** ;
- un seul périmètre : la France métropolitaine ;
- les données sont **définitives** jusqu'en 2024, et seulement **consolidées** depuis 2025.
  Les données consolidées peuvent encore être légèrement corrigées par RTE : c'est une limite
  à signaler dans le rapport. Je garde la colonne `nature` jusqu'au bout pour la retrouver ;
- 2026 est une année **incomplète**.

In [ ]:
# À quelles minutes la consommation est-elle renseignée ?
# Je prends les minutes dans la colonne "heure" (format "HH:MM").
minutes = brut["heure"].str[-2:]
pd.crosstab(minutes, brut["consommation"].notna(), margins=True).rename(
    columns={False: "consommation vide", True: "consommation remplie"}
)

**Ce que j'observe :** il y a 4 lignes par heure (:00, :15, :30, :45), mais la consommation
n'est remplie qu'à **:00 et :30**. Les lignes :15 et :45 ne servent qu'aux prévisions de RTE.
Les données de consommation sont donc **au pas de 30 minutes**.

## 3. Sélection de la consommation nationale

Je fais trois choix, l'un après l'autre, en comptant les lignes à chaque fois.

### 3a. Les colonnes : je garde seulement la consommation et ses repères

Je garde `nature`, `date`, `heure`, `date_heure` et `consommation`. Les 31 autres colonnes
sont interdites (voir l'étape 2).

### 3b. La période : du 1er décembre 2015 au 31 décembre 2025

**Il faut distinguer deux périodes :**

| | Dates | Rôle |
|---|---|---|
| **Période étudiée** | du 1er janvier 2016 au 31 décembre 2025 | les jours qu'on **prévoit** et sur lesquels on calcule les erreurs |
| **Période chargée** | du 1er décembre 2015 au 31 décembre 2025 | les données qu'on **garde**, période étudiée + 1 mois avant |

**Pourquoi ce mois de décembre 2015 en plus ?** Pour prévoir un jour, on a besoin de la
consommation des **jours d'avant** (ce sont les « retards »). Le plus loin qu'on remonte, c'est
**28 jours** : le benchmark B2 fait la moyenne des **4 mêmes jours de la semaine précédents**
(4 semaines × 7 jours = 28 jours).

**Exemple concret :** le premier jour à prévoir est le **vendredi 1er janvier 2016**.
- Le benchmark B2 a besoin des 4 vendredis d'avant : **25, 18, 11 et 4 décembre 2015**.
- Le 4 décembre 2015 est 28 jours avant le 1er janvier 2016.
- La prévision se fait le **jeudi 31 décembre 2015 à 14 h**, avec la consommation de ce
  jeudi matin.

Sans décembre 2015, on ne pourrait pas faire ces calculs pour les premiers jours de 2016. On
devrait soit les supprimer, soit les calculer autrement que les autres jours. Les données de
décembre 2015 servent donc **seulement d'entrées** (de retards). On ne les prévoit jamais et
on ne calcule jamais d'erreur dessus.

**Pourquoi s'arrêter en 2025 :** 2026 est incomplète et encore en données consolidées
(décision 3 dans `docs/decisions.md`).

**Pourquoi ne pas commencer en 2012 :** dix ans suffisent (7 pour apprendre, 1 pour valider,
2 pour tester), et les vieilles années ressemblent moins au niveau de consommation actuel.

Ces dates ne sont pas écrites à la main ici : elles viennent de `src/config.py`
(`DATA_DEBUT` et `DATA_FIN`), pour que tout le groupe utilise les mêmes.

### 3c. Les lignes vides : j'enlève les lignes :15 et :45

**Est-ce qu'on perd de l'information ? Non.** Sur les lignes de :15 et :45, la case
consommation est **toujours vide**. Le tableau de l'étape 2 le montre : 0 consommation remplie
à :15 et 0 à :45, sur tout le jeu de données. En retirant ces lignes, je ne retire donc
**aucune valeur de consommation**.

**Pourquoi ces lignes existent :** RTE range dans le même fichier deux choses qui n'ont pas le
même rythme :
- la **consommation réelle**, mesurée toutes les **30 minutes** (à :00 et :30) ;
- ses **propres prévisions** (`prevision_j1`, `prevision_j`), publiées toutes les **15 minutes**.

Pour loger les prévisions, le fichier a une ligne par quart d'heure. Les lignes de :15 et :45
ne contiennent que ces prévisions de RTE, et toutes les autres colonnes y sont vides.

**Et les prévisions de RTE ?** On ne s'en sert pas comme variables : ce serait recopier la
prévision des experts de RTE au lieu de faire la nôtre. Elles ne sont pas perdues pour autant :
elles restent dans le fichier brut (`data/donnees-brutes/rte/`), qu'on ne modifie jamais. On
pourra les y reprendre si on veut comparer nos résultats aux leurs.

**Ce que ça change pour la suite :** notre consommation est au **pas de 30 minutes**. À
l'étape 6, on fera donc la moyenne de **2 valeurs par heure** (:00 et :30), et pas de 4.

In [ ]:
# Le code de la sélection (dans src/rte.py) : 3a, 3b et 3c en une seule fonction
voir_code(rte.selectionner)

In [ ]:
conso = rte.selectionner(brut)  # période par défaut : config.DATA_DEBUT -> config.DATA_FIN

# Je compte ce que chaque choix a enlevé
dans_la_periode = (brut["date"] >= str(config.DATA_DEBUT)) & (brut["date"] <= str(config.DATA_FIN))
print(f"3a. Colonnes : {brut.shape[1]} -> {len(rte.COLONNES_GARDEES)} "
      f"(puis 5 : 'perimetre' ne sert plus une fois la France vérifiée)")
print(f"3b. Période du {config.DATA_DEBUT} au {config.DATA_FIN} : {len(brut):,} -> {dans_la_periode.sum():,} lignes")
print(f"3c. Lignes avec une consommation : {dans_la_periode.sum():,} -> {len(conso):,} (seulement :00 et :30)")
conso.head()

In [ ]:
# Vérification : la première et la dernière date gardées, et décembre 2015 bien présent
print("Première date :", conso["date"].min(), "| Dernière date :", conso["date"].max())
print("Jours de décembre 2015 gardés comme marge :", conso.loc[conso["date"] < "2016-01-01", "date"].nunique())

## 4. Nettoyage

**Ce que je fais :**
1. Je convertis `date_heure` en vraie date **UTC** et `consommation` en nombre.
2. Je cherche les **doublons** : deux lignes pour le même instant.
3. Je supprime les **lignes fantômes** des jours de changement d'heure.

**Le problème des lignes fantômes :** le dernier dimanche de mars, à 2 h du matin, les
horloges passent directement à 3 h. L'heure locale « 02:00 » **n'existe pas** ce jour-là.
Pourtant RTE fournit des lignes « 02:00 » et « 02:30 » : ce sont des **copies** d'autres
valeurs, qui tombent sur le même instant UTC que les vraies lignes « 03:00 » et « 03:30 ».

**Ma règle :** pour chaque ligne, je recalcule l'heure de Paris à partir de l'instant UTC.
Si elle ne correspond pas à l'heure écrite par RTE (`date` + `heure`), la ligne est fantôme et
je la supprime. Cette règle est générale : elle ne dépend pas d'une liste de dates écrite à la main.

In [ ]:
# 1-2. Les doublons : plusieurs lignes pour le même instant UTC
instant_utc = pd.to_datetime(conso["date_heure"], utc=True)
doublons = conso[instant_utc.duplicated(keep=False)]
print(f"{len(doublons)} lignes partagent leur instant UTC avec une autre ligne")
doublons.head(8)

In [ ]:
# 3. Le code qui repère les lignes fantômes (dans src/rte.py)
voir_code(rte.lignes_fantomes)

In [ ]:
fantome = rte.lignes_fantomes(conso)
print(f"{fantome.sum()} lignes fantômes trouvées, sur ces jours :")
print(sorted(conso.loc[fantome, "date"].unique()))
conso.loc[fantome, ["date", "heure", "date_heure", "consommation"]].head(6)

In [ ]:
# Je supprime les fantômes. La fonction vérifie aussi qu'il ne reste plus aucun doublon
# (sinon elle s'arrête avec une erreur) et met l'instant UTC en index.
conso_30min = rte.nettoyer(conso)
print(f"{len(conso_30min):,} demi-heures après nettoyage")
conso_30min.head()

## 5. Gestion des dates et heures

**Deux horloges :**
- **UTC** : une horloge sans changement d'heure. Chaque jour compte exactement 24 h.
  Je **stocke** toujours les données en UTC : aucun instant en double, aucun instant manquant.
- **Heure de Paris** : l'heure de la vie réelle (UTC+1 l'hiver, UTC+2 l'été). Le problème est
  posé dans cette heure-là (« à 14 h le jour J », « les 24 heures de J+1 »). Je la **calcule**
  quand j'en ai besoin.

**Ce que je fais :** je construis la **grille complète** de toutes les demi-heures de la
période, puis je place mes données dessus. Une demi-heure absente des données apparaît alors
comme une valeur manquante (NaN), au lieu de disparaître sans qu'on le voie.

**Conséquence des changements d'heure en heure de Paris :**
- dernier dimanche de **mars** : un jour de **23 h** (l'heure 2 h n'existe pas) ;
- dernier dimanche d'**octobre** : un jour de **25 h** (l'heure 2 h existe deux fois).

In [ ]:
# Le code de la grille (dans src/rte.py)
voir_code(rte.grille_30min)
voir_code(rte.mettre_sur_grille)

In [ ]:
# Début : 1er décembre 2015 à 0 h (Paris). Fin : 31 décembre 2025 à 23 h 30 (Paris).
grille = rte.grille_30min()
print("Grille :", grille[0], "->", grille[-1], f"({len(grille):,} demi-heures)")

conso_30min = rte.mettre_sur_grille(conso_30min)
manquantes = conso_30min["consommation_MW"].isna()
print(f"Demi-heures manquantes : {manquantes.sum()}")

In [ ]:
# Où sont les demi-heures manquantes ? Je les affiche en heure de Paris pour mieux comprendre.
trous = conso_30min.index[manquantes]
pd.DataFrame({
    "instant UTC": trous,
    "heure de Paris": trous.tz_convert(FUSEAU),
}).head(20)

**Ce que j'observe :** les trous tombent les jours de **passage à l'heure d'hiver** (fin
octobre), vers 2 h du matin. RTE ne fournit qu'une seule fois l'heure « 02:00 », alors qu'elle
existe deux fois ce jour-là. Il manque donc une heure en UTC. Je les traiterai à l'étape 7.

In [ ]:
# Je garde une copie de cette étape intermédiaire (30 minutes, UTC, grille complète)
rte.FICHIER_INTERIM.parent.mkdir(parents=True, exist_ok=True)
conso_30min.to_csv(rte.FICHIER_INTERIM)

## 6. Agrégation à l'heure

**Ce que je fais :** je passe de 2 valeurs par heure à 1 valeur par heure, en faisant la
**moyenne** des deux demi-heures.

**Pourquoi la moyenne et pas la somme :** la consommation est en **MW**, c'est une
**puissance** (un débit d'énergie), pas une quantité d'énergie. Si la puissance vaut
60 000 MW pendant la première demi-heure et 62 000 MW pendant la seconde, la puissance moyenne
sur l'heure est 61 000 MW. La somme (122 000) n'aurait pas de sens.

**Convention d'étiquette :** la valeur étiquetée **H** est la moyenne de H:00 et H:30,
c'est-à-dire la tranche **[H, H+1[**. C'est la convention de la décision 5 : la valeur
étiquetée 12 couvre 12 h-13 h, et c'est la dernière complète à 14 h.

**Si une des deux demi-heures manque :** je garde l'autre (meilleure estimation disponible),
et je compte combien de fois cela arrive. Si les deux manquent, l'heure reste vide. En pratique,
le tableau ci-dessous montre que ça n'arrive jamais : soit les 2 demi-heures sont là, soit
aucune (les heures d'octobre).

In [ ]:
# Le code du passage à l'heure (dans src/rte.py)
voir_code(rte.agreger_a_l_heure)

In [ ]:
# Combien de demi-heures sont disponibles dans chaque heure ? (0, 1 ou 2)
nb_demi_heures = conso_30min["consommation_MW"].resample("1h").count()
print("Nombre de demi-heures disponibles par heure :")
print(nb_demi_heures.value_counts().sort_index().rename("nombre d'heures"))

conso_h = rte.agreger_a_l_heure(conso_30min)
print(f"\n{len(conso_h):,} heures, de {conso_h.index[0]} à {conso_h.index[-1]} (UTC)")
conso_h.head()

In [ ]:
# Vérification à la main sur une heure : la moyenne des deux demi-heures
instant = pd.Timestamp("2019-01-15 18:00", tz=FUSEAU).tz_convert("UTC")
deux_demi_heures = conso_30min.loc[instant: instant + pd.Timedelta(minutes=30), "consommation_MW"]
print(deux_demi_heures)
print("Moyenne attendue :", deux_demi_heures.mean(), "| Valeur horaire :", conso_h.loc[instant, "consommation_MW"])

## 7. Gestion des valeurs manquantes

**Ma règle (décision 10) :**
- un trou de **3 heures de suite ou moins** est comblé par **interpolation linéaire** (une
  ligne droite entre la valeur d'avant et la valeur d'après) ;
- un trou plus long reste vide, car inventer plus de 3 heures serait trop risqué ;
- chaque heure comblée est marquée dans une colonne **`interpole`** (vrai / faux), pour qu'on
  puisse toujours la retrouver.

**Attention, piège de pandas :** `interpolate(limit=3)` ne fait pas ce que je veux. Sur un trou
de 5 heures, il remplit les 3 premières et laisse les 2 dernières. Je mesure donc d'abord la
**longueur de chaque trou**, et je n'interpole que les trous courts.

**Lien avec la fuite d'information :** l'interpolation utilise la valeur **d'après**. Ici, ce
n'est pas grave : les trous sont la nuit, vers 2 h, et la valeur d'après (vers 3 h) est connue
bien avant 14 h. Grâce à la colonne `interpole`, on pourra quand même le vérifier lors du test
de non-fuite.

In [ ]:
# Le code qui mesure les trous puis comble les courts (dans src/rte.py)
voir_code(rte.longueur_des_trous)
voir_code(rte.combler_les_trous)

In [ ]:
# Combien de trous, et de quelle longueur ?
longueur = rte.longueur_des_trous(conso_h["consommation_MW"])
# Une heure vide appartient à un trou de longueur L : il y a donc (nombre d'heures vides de longueur L) / L trous
heures_vides = longueur[longueur > 0].value_counts().sort_index()
print("Longueur du trou (en heures) -> nombre de trous :")
print((heures_vides / heures_vides.index).astype(int).rename("nombre de trous"))

In [ ]:
# Je comble les trous de config.INTERPOLATION_MAX_HEURES heures ou moins (3 h)
conso_h = rte.combler_les_trous(conso_h)

print(f"Heures interpolées : {conso_h['interpole'].sum()}")
print(f"Heures encore vides : {conso_h['consommation_MW'].isna().sum()}")

In [ ]:
# Je regarde une interpolation en heure de Paris : la nuit du passage à l'heure d'hiver 2016
exemple = conso_h.loc["2016-10-29 21:00":"2016-10-30 05:00"].copy()
exemple.index = exemple.index.tz_convert(FUSEAU)
exemple

## 8. Contrôles qualité

**Ce que je fais :** je vérifie automatiquement ce qui **doit** être vrai, avec la fonction
`rte.controler`. Si un contrôle échoue, elle s'arrête avec un message d'erreur clair : je ne
peux pas sauvegarder des données fausses sans m'en rendre compte. Les contrôles 1 à 4 sont
dans la fonction ; le contrôle 5 (les sauts) demande de regarder, donc je le fais ici.

1. Une ligne par heure, sans doublon ni heure sautée.
2. Plus aucune valeur manquante.
3. Des valeurs plausibles pour la France (entre 25 000 et 100 000 MW).
4. Le bon nombre d'heures par jour en heure de Paris : 24, sauf 23 en mars et 25 en octobre.
5. Pas de saut brutal d'une heure à l'autre (sinon je regarde ce qui se passe).

In [ ]:
# Le code des contrôles 1 à 4 (dans src/rte.py)
voir_code(rte.controler)

In [ ]:
rte.controler(conso_h)  # s'arrête avec une erreur si un contrôle échoue
print("Contrôles 1 à 4 : OK")

serie = conso_h["consommation_MW"]
print(f"\nMinimum : {serie.min():,.0f} MW le {serie.idxmin().tz_convert(FUSEAU)}")
print(f"Maximum : {serie.max():,.0f} MW le {serie.idxmax().tz_convert(FUSEAU)}")

In [ ]:
# 4. Détail : nombre d'heures par jour, en heure de Paris
heures_par_jour = rte.heures_par_jour(conso_h)
print(heures_par_jour.value_counts().rename("nombre de jours"))
print("\nJours de 23 h :", [str(j) for j in heures_par_jour[heures_par_jour == 23].index])
print("Jours de 25 h :", [str(j) for j in heures_par_jour[heures_par_jour == 25].index])

In [ ]:
# 5. Sauts brutaux : variation de plus de 8 000 MW en une heure.
# Ce n'est pas forcément une erreur, mais je veux les voir pour décider.
variation = serie.diff().abs()
sauts = variation[variation > 8_000]
print(f"{len(sauts)} sauts de plus de 8 000 MW\n")

# Pour chaque saut, j'affiche les heures autour (en heure de Paris) :
# si la valeur monte et reste haute, c'est un vrai phénomène ; si elle redescend aussitôt, c'est suspect.
for instant in sauts.index:
    autour = serie.loc[instant - pd.Timedelta("3h"): instant + pd.Timedelta("2h")]
    autour.index = autour.index.tz_convert(FUSEAU)
    print(autour.round(0), "\n")

**Ce que j'observe :** deux sauts, tous les deux un **dimanche soir de fin février** (2018 et
2019), entre 18 h et 19 h. La consommation reste haute après le saut : ce n'est pas une erreur
de mesure. C'est le moment où la nuit tombe et où l'éclairage et le chauffage se rallument, en
plein hiver. Je **garde** ces valeurs.

In [ ]:
# Bilan par année : nombre d'heures, heures interpolées, version des données
bilan = conso_h.assign(annee=conso_h.index.tz_convert(FUSEAU).year).groupby("annee").agg(
    heures=("consommation_MW", "size"),
    interpolees=("interpole", "sum"),
    moyenne_MW=("consommation_MW", "mean"),
    version=("nature", lambda v: ", ".join(sorted(v.unique()))),
)
bilan.round(0)

## 9. Exploration de la série

**Ce que je cherche :** les structures que les modèles devront reproduire.
- une **saisonnalité annuelle** (hiver / été) ;
- un **cycle hebdomadaire** (semaine / week-end) ;
- un **cycle journalier** (nuit / journée / pointe du soir) ;
- des **ruptures** : Covid en 2020, crise de l'énergie et sobriété fin 2022.

Pour l'exploration, je passe en **heure de Paris**, puisque la vie des gens suit l'heure locale.

In [ ]:
# Je travaille sur une copie en heure de Paris pour l'exploration
explo = conso_h[["consommation_MW"]].copy()
explo.index = explo.index.tz_convert(FUSEAU)
explo["heure"] = explo.index.hour
explo["jour_semaine"] = explo.index.dayofweek  # 0 = lundi, 6 = dimanche
explo["mois"] = explo.index.month
explo["annee"] = explo.index.year

In [ ]:
# Graphique 1 : toute la période, en moyenne journalière (sinon 88 000 points illisibles)
moyenne_jour = explo["consommation_MW"].resample("D").mean() / 1000  # en GW, plus lisible

fig, ax = plt.subplots()
ax.plot(moyenne_jour.index, moyenne_jour, color=BLEU, linewidth=0.8)
ax.set_title("Consommation moyenne journalière, France (GW)")
ax.set_ylabel("GW")
plt.show()

**Ce que je lis :** une très forte **saisonnalité annuelle** : beaucoup plus de consommation
en hiver (chauffage électrique), moins en été (environ 40 GW). Les pics d'hiver varient beaucoup
d'une année à l'autre selon le froid : le pic le plus haut est fin **février 2018** (vague de
froid). Le printemps 2020 descend plus bas que d'habitude (**confinement**).

In [ ]:
# Graphique 2 : moyenne par année (Décembre 2015 seul n'est pas une année complète : je l'enlève)
par_annee = explo.loc[explo["annee"] >= 2016].groupby("annee")["consommation_MW"].mean() / 1000

fig, ax = plt.subplots(figsize=(9, 3.5))
# Je trace des points reliés et pas des barres : une barre doit partir de 0, et à partir de 0
# on ne verrait plus les différences entre années (quelques GW sur environ 50).
ax.plot(par_annee.index, par_annee, color=BLEU, marker="o", markersize=7)
ax.set_xticks(par_annee.index)
ax.set_title("Consommation moyenne par année (GW)")
ax.set_ylabel("GW")
for x, y in zip(par_annee.index, par_annee):
    ax.annotate(f"{y:.1f}", (x, y), textcoords="offset points", xytext=(0, 8),
                ha="center", fontsize=9, color=GRIS_TEXTE)
plt.show()

**Ce que je lis :** environ 54-55 GW en moyenne de 2016 à 2019, puis une chute en **2020**
(Covid, 50,8 GW), une remontée en 2021, et de nouveau une baisse en **2022-2023** (prix de
l'énergie, appels à la sobriété) : 49,7 GW en 2023, le plus bas de la période. 2024 et 2025
restent à ce **niveau bas**. C'est une **rupture de niveau** importante : un modèle appris
surtout sur 2016-2019 risque de **surestimer** la consommation de 2023 à 2025. C'est un argument
pour la **réestimation mensuelle** (décision 4), et un point à surveiller dans les erreurs.

In [ ]:
# Graphique 3 : deux semaines en heure par heure (janvier 2019), pour voir les cycles
deux_semaines = explo.loc["2019-01-07":"2019-01-20", "consommation_MW"] / 1000

fig, ax = plt.subplots()
ax.plot(deux_semaines.index, deux_semaines, color=BLEU)
ax.set_title("Deux semaines de janvier 2019, heure par heure (GW)")
ax.set_ylabel("GW")
plt.show()

**Ce que je lis :** chaque jour se ressemble (cycle **journalier**), et le samedi et le
dimanche sont plus bas (cycle **hebdomadaire**). C'est ce qui justifie les benchmarks
« même jour de la semaine précédente » (retard de 168 h).

In [ ]:
# Graphique 4 : profil moyen d'une journée, selon le type de jour
type_jour = np.select(
    [explo["jour_semaine"] <= 4, explo["jour_semaine"] == 5],
    ["Lundi à vendredi", "Samedi"],
    default="Dimanche",
)
profil = explo.groupby([type_jour, "heure"])["consommation_MW"].mean().unstack(0) / 1000

fig, ax = plt.subplots(figsize=(9, 4))
for nom, couleur in [("Lundi à vendredi", BLEU), ("Samedi", ORANGE), ("Dimanche", VERT)]:
    ax.plot(profil.index, profil[nom], color=couleur, marker="o", markersize=4, label=nom)
ax.set_xticks(range(0, 24, 2))
ax.set_xlabel("Heure de Paris")
ax.set_ylabel("GW")
ax.set_title("Profil moyen d'une journée selon le type de jour")
ax.legend(frameon=False)
plt.show()

**Ce que je lis :**
- creux vers **4-5 h**, montée le matin, plateau autour de **12 h**, deuxième pointe vers **19 h** ;
- une remontée à **23 h** : c'est le passage en **heures creuses** (chauffe-eau qui démarrent) ;
- en semaine, la journée est bien plus haute (environ 59-60 GW à midi contre 52-54 GW le week-end) ;
- la nuit, samedi et semaine se ressemblent ; le dimanche est le plus bas presque toute la journée.

La forme de la journée change donc selon le jour. Donc le modèle doit
connaître **l'heure** et **le jour de la semaine**, et l'effet de l'heure n'est pas le même
selon le jour. C'est un argument pour un modèle **par heure**, avec le calendrier.

In [ ]:
# Graphique 5 : répartition de la consommation journalière par mois
explo_jour = explo["consommation_MW"].resample("D").mean().to_frame() / 1000
explo_jour["mois"] = explo_jour.index.month
donnees_mois = [explo_jour.loc[explo_jour["mois"] == m, "consommation_MW"] for m in range(1, 13)]

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot(donnees_mois, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor="#d6e6f8", edgecolor=BLEU),
           medianprops=dict(color=BLEU, linewidth=2),
           whiskerprops=dict(color=BLEU), capprops=dict(color=BLEU),
           flierprops=dict(marker="o", markersize=3, markeredgecolor=BLEU))
ax.set_xticks(range(1, 13), ["janv", "févr", "mars", "avr", "mai", "juin",
                             "juil", "août", "sept", "oct", "nov", "déc"])
ax.set_ylabel("GW (moyenne du jour)")
ax.set_title("Consommation journalière selon le mois")
ax.grid(axis="x", visible=False)
plt.show()

**Ce que je lis :** les mois d'hiver sont plus hauts (médiane d'environ 68 GW en janvier contre
42 GW en août), mais surtout **beaucoup plus dispersés** (environ 50 à 87 GW en janvier) :
en hiver, la consommation dépend fortement de la température (un jour froid fait monter la
consommation). En été, elle est plus stable. C'est pour ça qu'on ajoutera la **météo** dans les
modèles, et qu'on s'attend à de plus grosses erreurs en hiver.

In [ ]:
# Les jours les plus atypiques : je regarde les 10 jours les plus bas par rapport à leur mois.
# (écart à la médiane des jours du même mois et de la même année)
mediane_mois = explo_jour.groupby([explo_jour.index.year, "mois"])["consommation_MW"].transform("median")
ecart = (explo_jour["consommation_MW"] - mediane_mois) / mediane_mois * 100
ecart.sort_values().head(10).round(1).rename("écart à la médiane du mois (%)")

**Ce que je lis :** les jours les plus bas par rapport à leur mois sont la période de **Noël
et du jour de l'an** (jusqu'à -32 % le 1er janvier 2023) et le week-end de la **Toussaint**
(1er et 2 novembre). Ce sont des jours de fête : le calendrier ordinaire (jour de la semaine,
mois) ne suffit pas à les expliquer. Il faudra des variables **fériés**, **ponts** et une
variable **période de Noël**.

**Limite de cette méthode :** je compare chaque jour à la médiane de son mois. Le confinement
de 2020 n'apparaît pas ici, car il a duré tout le mois d'avril : la médiane d'avril 2020 est
elle-même basse. Pour le voir, il faut comparer avec les autres années (graphique 1).

## 10. Sauvegarde des données préparées

**Ce que je sauvegarde :** `data/donnees-preparees/rte/conso_horaire_utc.csv`, avec :

| Colonne | Ce qu'elle contient | Exemple |
|---|---|---|
| `date_heure_utc` | le **début** de l'heure, en UTC. C'est l'index : une ligne par heure | `2019-01-15 17:00 UTC` = 18 h à Paris |
| `consommation_MW` | la consommation **moyenne** de la France pendant cette heure, en MW. « [H, H+1[ » veut dire « de H h pile jusqu'à H+1 h, sans compter H+1 h pile » | 75 765 MW = moyenne de 18 h 00 (74 729) et 18 h 30 (76 801) |
| `interpole` | `True` (vrai) si **j'ai fabriqué** la valeur à l'étape 7, parce que RTE ne la donnait pas. `False` (faux) si c'est une vraie mesure de RTE | `True` pour seulement 10 heures : 2 h du matin les dimanches de passage à l'heure d'hiver |
| `nature` | la version de la donnée chez RTE | « Données définitives » (jusqu'en 2024) ou « Données consolidées » (2025) |

**Pourquoi garder la colonne `interpole` :** ces 10 valeurs ne sont pas des mesures, ce sont
mes estimations (une ligne droite entre l'heure d'avant et l'heure d'après). En gardant la
trace, on peut à tout moment :
- les **retirer du calcul des erreurs** (on ne note pas un modèle sur une valeur inventée) ;
- vérifier qu'elles ne créent pas de **fuite d'information** (l'interpolation utilise l'heure
  d'après) ;
- les **citer dans le rapport** pour être transparent.

**Pourquoi en UTC :** c'est le format d'échange décidé avec Khadim (décision 16) : la météo
SYNOP est aussi en UTC, donc on pourra joindre les deux fichiers directement. L'heure de Paris
se recalcule en une ligne : `index.tz_convert("Europe/Paris")`.

In [ ]:
chemin = rte.sauvegarder(conso_h)
print("Enregistré :", chemin.relative_to(RACINE))

# Je relis le fichier pour vérifier qu'il est identique à ce que j'ai en mémoire
relu = rte.lire_prepare(chemin)
assert len(relu) == len(conso_h)
assert relu.index.tz is not None and str(relu.index.tz) == "UTC"
assert np.allclose(relu["consommation_MW"], conso_h["consommation_MW"])
print(f"Relecture OK : {len(relu):,} heures, du {relu.index[0]} au {relu.index[-1]} (UTC)")
relu.head()

## 11. Disponibilité des données et protocole à 14 h

**La règle du projet :** à 14 h le jour J, on ne peut utiliser que les informations
**réellement disponibles à ce moment-là**.

Cette section ne construit pas de modèle. Elle **documente** ce qu'on sait de la donnée RTE à
14 h, pour éviter une fuite d'information plus tard. Le piège typique : créer une variable
« consommation de la veille » qui utiliserait des heures **pas encore publiées** au moment où
la prévision est censée être faite.

Je réponds à six questions :
1. Quelle variable RTE est observée ?
2. À quelle fréquence est-elle publiée ?
3. Quelle profondeur d'historique est disponible ?
4. Quelles observations du jour J sont réellement disponibles à 14 h ?
5. Qu'est-ce qui est indisponible à 14 h ?
6. Les données sont-elles corrigées après coup (consolidation) ?

### 11.1 La variable observée

| | |
|---|---|
| **Variable** | `consommation` : la consommation d'électricité de la France métropolitaine |
| **Unité** | MW (mégawatts) : une **puissance**, c'est-à-dire un débit d'énergie à un instant donné |
| **Source** | RTE, application éCO2mix |
| **Comment elle est obtenue** | télémesures des ouvrages du réseau, complétées par des estimations (temps réel), puis par les comptages (consolidé, définitif) |
| **Notre cible** | la moyenne horaire de cette consommation, pour chacune des 24 heures de J+1 |

### 11.2 Fréquence de publication : une même donnée, trois versions

La même consommation est publiée **trois fois** par RTE, de plus en plus précise avec le temps.
Ce sont les descriptions officielles des jeux de données sur ODRÉ :

| Version | Pas de temps | Quand est-elle publiée ? | Jeu de données |
|---|---|---|---|
| **Temps réel** | 15 minutes | en continu, **mise à jour toutes les 15 minutes**, avec un petit retard | `eco2mix-national-tr` |
| **Consolidée** | 30 minutes | **au milieu du mois suivant** (M+1), après vérification | `eco2mix-national-cons-def` |
| **Définitive** | 30 minutes | **au second semestre de l'année suivante** (A+1), quand tous les comptages sont vérifiés | `eco2mix-national-cons-def` |

**Ce que ça veut dire pour nous :** à 14 h le jour J, la consommation du jour J et des jours
récents n'existe **qu'en version temps réel**. Les versions consolidée et définitive arrivent
des semaines ou des mois plus tard.

Je mesure ci-dessous le retard de publication du temps réel : je demande à l'API la dernière
valeur publiée et je la compare à l'heure qu'il est. Le résultat change à chaque exécution :
c'est une mesure ponctuelle, pas une garantie.

In [ ]:
URL_TEMPS_REEL = (
    "https://odre.opendatasoft.com/api/explore/v2.1/catalog/datasets/"
    "eco2mix-national-tr/records"
)
try:
    reponse = requests.get(URL_TEMPS_REEL, params={
        "select": "date_heure, consommation",
        "where": "consommation is not null",
        "order_by": "date_heure desc",  # la plus récente en premier
        "limit": 1,
    }, timeout=30)
    reponse.raise_for_status()
    derniere = reponse.json()["results"][0]

    maintenant = pd.Timestamp.now(tz=FUSEAU)
    debut_quart = pd.Timestamp(derniere["date_heure"]).tz_convert(FUSEAU)
    fin_quart = debut_quart + pd.Timedelta(minutes=15)  # la valeur couvre 15 minutes

    print(f"Heure actuelle (Paris)          : {maintenant:%d/%m/%Y %H:%M}")
    print(f"Dernière valeur publiée (Paris) : quart d'heure {debut_quart:%H:%M}-{fin_quart:%H:%M}, "
          f"{derniere['consommation']:,} MW")
    retard = (maintenant - fin_quart).total_seconds() / 60
    print(f"Retard : la dernière valeur publiée s'est terminée il y a {retard:.0f} minutes")
except Exception as erreur:
    # Si pas d'internet, je n'arrête pas le notebook : cette mesure est seulement informative
    print("Mesure impossible (pas de connexion ?) :", erreur)

**Ce que j'en retiens :** le samedi 3 octobre 2026 vers 14 h 40, la dernière valeur publiée
était celle de 14 h 15-14 h 30. Elle était donc en ligne **une dizaine de minutes** après la fin
de son quart d'heure (11 et 13 minutes sur deux mesures). Une valeur de consommation n'est
donc **pas** disponible dès qu'elle est mesurée : il faut compter un retard d'au moins un quart
d'heure.

**Limite :** deux mesures, le même jour, un samedi. Il faudrait refaire la mesure plusieurs
jours, vers 14 h, en semaine, pour confirmer. Notre règle de 11.4 garde de toute façon une
marge d'une heure.

### 11.3 Profondeur d'historique

| Version | Période disponible aujourd'hui |
|---|---|
| Définitive | du 1er janvier 2012 au 31 décembre 2024 |
| Consolidée | du 1er janvier 2025 au 30 juin 2026 |
| Temps réel | seulement **depuis la dernière consolidation** (juillet 2026) : RTE remplace ensuite ces valeurs par les versions consolidées |

Nous utilisons du 1er décembre 2015 au 31 décembre 2025 (étape 3) : un historique largement
suffisant.

### 11.4 Ce qu'on connaît du jour J à 14 h

**Notre règle (décision 5) :** à 14 h, la dernière heure **complète et publiée** est la tranche
**12 h-13 h** (la valeur horaire étiquetée 12).

**Pourquoi pas la tranche 13 h-14 h ?**
- elle se termine **à 14 h pile** : à 14 h, elle vient juste de finir ;
- sa dernière demi-heure (13 h 30-14 h) n'est **pas encore publiée** à 14 h (il y a le retard
  de publication mesuré plus haut) ;
- notre valeur horaire est une **moyenne de deux demi-heures** : avec une seule des deux, elle
  serait incomplète.

**Ce qu'on perd :** la demi-heure 13 h-13 h 30 est probablement publiée vers 13 h 45-14 h. On
choisit de ne pas l'utiliser : c'est une **marge de sécurité**. On perd un peu d'information,
mais on est sûr de ne jamais utiliser une valeur pas encore publiée.

**Attention à l'heure :** nos données sont en UTC. 14 h à Paris, c'est 13 h UTC en hiver et
12 h UTC en été. Le code ci-dessous calcule tout en heure de Paris, puis convertit.

In [ ]:
# La règle est codée dans src/protocole.py. Les variables des modèles devront toutes passer par là.
voir_code(protocole.fin_des_donnees_connues)
voir_code(protocole.est_connue_a_14h)

In [ ]:
from src.protocole import est_connue_a_14h, fin_des_donnees_connues

# Exemple sur un jour d'hiver et un jour d'été
lignes = []
for jour_J in ["2023-01-10", "2023-07-11"]:
    origine = pd.Timestamp(f"{jour_J} {config.HEURE_ORIGINE}:00", tz=FUSEAU)
    fin = fin_des_donnees_connues(jour_J)
    lignes.append({
        "jour J": jour_J,
        "prévision faite à (Paris)": f"{origine:%H:%M}",
        "prévision faite à (UTC)": f"{origine.tz_convert('UTC'):%H:%M}",
        "dernière heure connue (Paris)": "12 h-13 h",
        "dernière heure connue (UTC)": f"{(fin - pd.Timedelta(hours=1)):%H:%M}-{fin:%H:%M}",
    })
pd.DataFrame(lignes)

**Ce que je lis :** la règle est la même en heure de Paris toute l'année (12 h-13 h), mais pas
en UTC : 11 h-12 h UTC en hiver, 10 h-11 h UTC en été. C'est pour ça qu'on ne doit **jamais**
écrire une règle du type « on garde jusqu'à 12 h UTC » : elle serait fausse une partie de
l'année.

### 11.5 Le piège de « la consommation de la veille »

On veut prévoir les 24 heures de **J+1**. La « veille » de J+1, c'est **J**, le jour où on
fait la prévision. Or à 14 h, le jour J n'est **connu que jusqu'à 13 h**.

Pour chaque heure H de J+1, je regarde quels « retards » sont autorisés :
- **retard 24 h** : la même heure, le jour J (la vraie veille) ;
- **retard 48 h** : la même heure, le jour J-1 (l'avant-veille) ;
- **retard 168 h** : la même heure, le jour J-6 (le même jour de la semaine, une semaine avant).

Je ne l'écris pas à la main : je le **calcule** avec la règle de 11.4.

In [ ]:
jour_J = pd.Timestamp("2023-01-10")  # un mardi ordinaire, sans changement d'heure

lignes = []
for H in range(24):
    # L'heure H du jour J+1 qu'on veut prévoir, puis les heures "retardées"
    cible = pd.Timestamp(f"{jour_J + pd.Timedelta(days=1):%Y-%m-%d} {H:02d}:00", tz=FUSEAU)
    ligne = {"heure cible H (J+1)": f"{H} h"}
    for nom, jours in [("retard 24 h (J)", 1), ("retard 48 h (J-1)", 2), ("retard 168 h (J-6)", 7)]:
        heure_retard = (cible - pd.Timedelta(days=jours)).tz_convert("UTC")
        ligne[nom] = "Oui" if est_connue_a_14h(heure_retard, f"{jour_J:%Y-%m-%d}") else "NON"
    lignes.append(ligne)

retards = pd.DataFrame(lignes).set_index("heure cible H (J+1)")
retards.T  # je retourne le tableau pour qu'il tienne en largeur

In [ ]:
# Je vérifie automatiquement ce que dit le protocole (docs/protocole.md) :
# - retard 24 h disponible seulement pour H <= 12 ;
# - retards 48 h et 168 h toujours disponibles.
dispo_24h = retards["retard 24 h (J)"] == "Oui"
assert list(dispo_24h[dispo_24h].index) == [f"{H} h" for H in range(13)]
assert (retards["retard 48 h (J-1)"] == "Oui").all()
assert (retards["retard 168 h (J-6)"] == "Oui").all()
print("Conforme au protocole : retard 24 h seulement pour H de 0 à 12 ; 48 h et 168 h toujours.")

**Ce que je lis :**
- pour prévoir **J+1 de 0 h à 12 h**, on peut utiliser la même heure du jour J : elle est
  déjà passée à 14 h ;
- pour prévoir **J+1 de 13 h à 23 h**, la même heure du jour J **n'existe pas encore** à 14 h.
  Utiliser la vraie « veille » pour ces heures serait une **fuite d'information**. On prend
  alors l'avant-veille (retard 48 h) à la place.

C'est exactement le piège : une variable `consommation.shift(24)` calculée naïvement sur
toute la série donnerait, pour J+1 à 18 h, la consommation de J à 18 h, qui n'était pas connue
à 14 h.

In [ ]:
# Illustration : ce que je sais à 14 h le mardi 10 janvier 2023, et ce que je dois prévoir
debut_vue = pd.Timestamp("2023-01-09", tz=FUSEAU)   # J-1
fin_vue = pd.Timestamp("2023-01-12", tz=FUSEAU)     # fin de J+1
origine = pd.Timestamp("2023-01-10 14:00", tz=FUSEAU)
debut_J1 = pd.Timestamp("2023-01-11", tz=FUSEAU)

vue = conso_h["consommation_MW"].copy()
vue.index = vue.index.tz_convert(FUSEAU)
vue = vue[(vue.index >= debut_vue) & (vue.index < fin_vue)] / 1000

connu = vue[[est_connue_a_14h(t.tz_convert("UTC"), "2023-01-10") for t in vue.index]]
cible = vue[vue.index >= debut_J1]
inconnu = vue[(vue.index > connu.index[-1]) & (vue.index < debut_J1)]

fig, ax = plt.subplots(figsize=(11, 4))
ax.step(connu.index, connu, where="post", color=BLEU, label="Connu à 14 h (on peut l'utiliser)")
ax.step(inconnu.index, inconnu, where="post", color="#a3a29c", linestyle="--",
        label="Pas encore publié à 14 h (interdit)")
ax.step(cible.index, cible, where="post", color=ORANGE, label="J+1 : les 24 heures à prévoir")
ax.axvline(origine, color=GRIS_TEXTE, linewidth=1)
ax.set_ylim(vue.min() - 2, vue.max() + 5)  # de la place en haut pour l'étiquette
ax.annotate("Prévision faite ici (mardi 14 h)", (origine, vue.max() + 4), xytext=(-6, 0),
            textcoords="offset points", ha="right", va="center", fontsize=9, color=GRIS_TEXTE)
ax.set_ylabel("GW")
ax.set_title("Ce qu'on sait à 14 h le jour J (exemple : mardi 10 janvier 2023)")
# La légende sous le graphique, pour ne pas cacher les courbes
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.show()

### 11.6 Ce qui est indisponible à 14 h le jour J

| Information | Pourquoi elle est interdite |
|---|---|
| Consommation du jour J à partir de 13 h | pas terminée ou pas encore publiée |
| Consommation de J+1 | c'est ce qu'on cherche à prévoir |
| Production, échanges, CO2 (les 31 autres colonnes d'éCO2mix) | mesurés en même temps que la consommation, donc inconnus pour J+1 |
| Versions consolidée et définitive des jours récents | publiées des semaines ou des mois plus tard |
| Prévisions de RTE | autorisées seulement pour **comparer** nos résultats, jamais comme variable d'entrée |

### 11.7 Les corrections après coup : une limite à écrire dans le rapport

Notre historique est en version **définitive** (jusqu'en 2024) et **consolidée** (2025). Or, à
14 h le jour J, un prévisionniste réel n'aurait eu que la version **temps réel** des jours
récents, un peu moins précise.

**Est-ce une fuite d'information ?** Ce n'est pas une fuite de **temps** : on n'utilise aucune
heure du futur. C'est une petite fuite de **qualité** : nos entrées sont un peu plus propres
que celles qu'on aurait eues en vrai. Nos erreurs seront donc probablement **un peu
optimistes**.

**Peut-on mesurer l'écart ?** Pas avec les données publiques : RTE efface la version temps
réel quand la version consolidée arrive, donc on ne peut pas comparer les deux versions sur une
même période. On l'écrit comme une **limite** dans le rapport (et dans
`docs/disponibilite_variables.md`).

### 11.8 Résumé du protocole d'information à 14 h

| Question | Réponse |
|---|---|
| Variable observée | Consommation nationale (MW), éCO2mix, RTE |
| Fréquence de publication | Temps réel : toutes les 15 min, en ligne une dizaine de minutes après la fin du quart d'heure (2 mesures). Consolidée : M+1. Définitive : A+1 |
| Historique disponible | Depuis 2012 ; nous utilisons décembre 2015 à décembre 2025 |
| Connu du jour J à 14 h | Jusqu'à la tranche **12 h-13 h** (heure de Paris) incluse |
| Indisponible à 14 h | Le jour J à partir de 13 h, tout J+1, les autres colonnes d'éCO2mix |
| Retards autorisés pour l'heure H de J+1 | 24 h seulement si H ≤ 12 ; 48 h et 168 h toujours |
| Corrections ultérieures | Oui (temps réel → consolidé → définitif). Notre historique est la version corrigée : limite à signaler |

## Résumé de ce notebook

| Étape | Décision |
|---|---|
| Source | API ODRÉ, jeu `eco2mix-national-cons-def` (définitif jusqu'en 2024, consolidé en 2025) |
| Colonnes | Uniquement la consommation (les autres colonnes sont interdites : fuite d'information) |
| Pas de temps | 30 min dans les données (lignes :00 et :30) → moyenne sur l'heure |
| Heures | Stockage en UTC ; heure de Paris recalculée pour l'analyse |
| Changement d'heure | Lignes fantômes de mars supprimées (20) ; heure manquante d'octobre interpolée (10) |
| Trous | Interpolation linéaire si 3 h ou moins, colonne `interpole` |
| Information à 14 h | Consommation connue jusqu'à la tranche 12 h-13 h (Paris) ; retard 24 h seulement pour H ≤ 12 |
| Limites à signaler | 2025 en données consolidées ; historique en version corrigée, plus propre que le temps réel disponible à 14 h |

**Où retrouver tout ça :**
- le code : `src/rte.py` (préparation) et `src/protocole.py` (règle des 14 h) ;
- relancer la préparation sans le notebook : `python -m src.rte` ;
- les tests automatiques (sans internet) : `tests/test_rte.py` et `tests/test_protocole.py`,
  lancés avec `pytest`.

**Prochaine étape :** les benchmarks (`src/benchmarks.py`), qui utiliseront ce fichier et la
règle des 14 h.